In [10]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/robiulhasanjisan/bangladesh-daraz-reviews-for-sentiment-analysis/Daraz_Master_Reviews_bd.csv


In [30]:


!pip install --quiet transformers datasets scikit-learn torch

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, f1_score
from transformers import AutoTokenizer, AutoModel, Trainer, TrainingArguments


data_path = "/kaggle/input/datasets/robiulhasanjisan/bangladesh-daraz-reviews-for-sentiment-analysis/Daraz_Master_Reviews_bd.csv"
df = pd.read_csv(data_path)
df = df[['review_text', 'rating']].dropna()
df['rating'] = df['rating'].round().astype(int)
df = df[df['rating'].between(1,5)]
print("Rating distribution:\n", df['rating'].value_counts())


train_texts, test_texts, train_labels, test_labels = train_test_split(
    df['review_text'].tolist(),
    df['rating'].tolist(),
    test_size=0.2,
    random_state=42,
    stratify=df['rating'].tolist()
)


MODEL_NAME = "sagorsarker/bangla-bert-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=128)
test_encodings = tokenizer(test_texts, truncation=True, padding=True, max_length=128)


class ReviewDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k,v in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx]-1)  # 0-index for 5-class
        return item

train_dataset = ReviewDataset(train_encodings, train_labels)
test_dataset = ReviewDataset(test_encodings, test_labels)


labels_np = np.array(train_labels)
class_counts = np.array([len(np.where(labels_np==i)[0]) for i in range(1,6)])
weights = 1.0 / class_counts
sample_weights = np.array([weights[label-1] for label in labels_np])
sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)
weights_tensor = torch.tensor(weights, dtype=torch.float)


class BertWithFocalLoss(nn.Module):
    def __init__(self, model_name, num_labels=5, alpha=None, gamma=2):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.classifier = nn.Linear(self.bert.config.hidden_size, num_labels)
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, input_ids=None, attention_mask=None, labels=None, token_type_ids=None, **kwargs):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids, **kwargs)
        logits = self.classifier(outputs.pooler_output)
        loss = None
        if labels is not None:
            # Move weights to the same device as labels
            ce_loss = F.cross_entropy(logits, labels, weight=self.alpha.to(labels.device))
            pt = torch.exp(-ce_loss)
            loss = ((1-pt)**self.gamma * ce_loss).mean()
        return {'loss': loss, 'logits': logits}

model = BertWithFocalLoss(MODEL_NAME, alpha=weights_tensor)


training_args = TrainingArguments(
    output_dir="/kaggle/working/bangla_sentiment",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    logging_steps=50,
    save_steps=500,
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_dir="/kaggle/working/logs",
    report_to="none"
)


def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average='macro')
    }


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)


trainer.get_train_dataloader = lambda: DataLoader(
    train_dataset, batch_size=training_args.per_device_train_batch_size, sampler=sampler
)


trainer.train()


preds = trainer.predict(test_dataset)
pred_labels = preds.predictions.argmax(-1) + 1  # Convert back to 1-5
print(classification_report(test_labels, pred_labels, digits=4))

Rating distribution:
 rating
5    7229
4     602
1     419
3     262
2     123
Name: count, dtype: int64


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sagorsarker/bangla-bert-base
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.
/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalar

Step,Training Loss
50,1.426131
100,1.169182
150,0.913035
200,0.774440
250,0.626955
300,0.647354
350,0.457882
400,0.364182
450,0.366046
500,0.268456


/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


              precision    recall  f1-score   support

           1     0.1198    0.5119    0.1941        84
           2     0.0081    0.0800    0.0148        25
           3     0.0418    0.3269    0.0741        52
           4     0.0619    0.3667    0.1059       120
           5     0.7500    0.0021    0.0041      1446

    accuracy                         0.0631      1727
   macro avg     0.1963    0.2575    0.0786      1727
weighted avg     0.6395    0.0631    0.0227      1727

